In [0]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType
)
from delta.tables import DeltaTable
import pandas as pd

In [0]:
display(
    pd.read_excel("/Volumes/catalog_southeastasia_mdm_pr/share_mdm_config/mdm_config_files/cdp_mdm_pr/touchpoint_retry/pr_touchpoint_loading_blob_config_20260618.xlsx")
)

In [0]:
# dbutils.widgets.text("Market_Conddition", "market in ['AUS'] ")

In [0]:


# =========================
# 1) 参数配置（按需修改）
# =========================

# market in ['a', 'b']
market_condition = dbutils.widgets.get("Market_Conddition")

EXCEL_PATH = "/Volumes/catalog_southeastasia_mdm_pr/share_mdm_config/mdm_config_files/cdp_mdm_pr/touchpoint_retry/pr_touchpoint_loading_blob_config_20260618.xlsx"

print(f"market_condition: {market_condition}")
print(f"EXCEL_PATH: {EXCEL_PATH}")


MYSQL_HOST = "mysqlflex-ap-southeastasia-prod-cepa-talend-02.mysql.database.azure.com"
MYSQL_USER = "talend_mdm_read_only@mysql-ap-southeastasia-prod-cepa-talend-02"
MYSQL_PASSWORD = "Q4&4c9JpMSb5A=xa"
MYSQL_PORT = "3306"
MYSQL_DRIVER = "com.mysql.cj.jdbc.Driver"
MYSQL_USE_SSL = True

# =========================
# 3) 读取 Excel
# =========================
# 依赖: pip install pandas openpyxl
pdf = pd.read_excel(EXCEL_PATH, engine="openpyxl").query("is_loading_blob_active == True").query(market_condition)
# .query("~source_database.str.contains('elcconsumermdm', na=False) ")
# .query(" source_database.str.contains('elcconsumermdm', na=False) ")

# 统一列名（去首尾空格）
pdf.columns = [str(c).strip() for c in pdf.columns]

required_cols = ["Index", "market", "source_database", "source_table", "target_path"]
missing = [c for c in required_cols if c not in pdf.columns]
if missing:
    raise ValueError(f"Excel 缺少必要列: {missing}")

# 只保留需要的列，并转成字符串（避免空值类型不一致）
pdf = pdf[required_cols].copy()
for c in required_cols:
    pdf[c] = pdf[c].astype("string")

input_rows = pdf.to_dict(orient="records")


display(pdf)


# =========================
# 4) 计数函数
# =========================

def build_jdbc_url(database: str) -> str:
    ssl_str = "&useSSL=true&enabledTLSProtocols=TLSv1.2" if MYSQL_USE_SSL else ""
    return (
        f"jdbc:mysql://{MYSQL_HOST}:3306/{database}"
        f"?serverTimezone=UTC&rewriteBatchedStatements=true&useUnicode=true&characterEncoding=UTF-8&zeroDateTimeBehavior=CONVERT_TO_NULL&useCompression=true{ssl_str}"
    )

def get_mysql_count(db_name: str, table_name: str):
    """
    返回: (count, error_msg)
    """
    if not db_name or not table_name:
        return None, "mysql database 或 source_table 为空"

    # 用子查询做 dbtable，避免直接拼接全库表名时 JDBC 兼容性问题
    sql = f"(SELECT COUNT(*) AS cnt FROM `{db_name}`.`{table_name}`) t"

    # jdbc_url = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{db_name}?useSSL=false&serverTimezone=UTC"
    jdbc_url = build_jdbc_url(db_name)
    try:
        cnt = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", sql)
            .option("user", MYSQL_USER)
            .option("password", MYSQL_PASSWORD)
            .option("driver", MYSQL_DRIVER)
            .load()
            .collect()[0]["cnt"]
        )
        return int(cnt), None
    except Exception as e:
        return None, str(e)


def get_delta_count(delta_path: str):
    """
    返回: (count, error_msg)
    """
    if not delta_path or str(delta_path).strip() == "":
        return None, "target_path 为空"

    p = str(delta_path).strip()
    try:
        # 先判断是否为可识别的 Delta 路径（也能覆盖路径不存在场景）
        if not DeltaTable.isDeltaTable(spark, p):
            return None, "路径不存在或不是 Delta 表"
        cnt = spark.read.format("delta").load(p).count()
        return int(cnt), None
    except Exception as e:
        # 包含路径不存在、权限问题、格式不正确等
        return None, str(e)




In [0]:
# =========================
# 5) 逐行比对并生成结果
# =========================
result_data = []

for i, r in enumerate(input_rows):
    
    db_name = None if pd.isna(r["source_database"]) else str(r["source_database"]).strip()
    table_name = None if pd.isna(r["source_table"]) else str(r["source_table"]).strip()
    delta_path = None if pd.isna(r["target_path"]) else str(r["target_path"]).strip()

    # if table_name != "slandtouchpoint":
    #     continue

    print(f"{i}: {db_name}.{table_name}")
    mysql_count, mysql_err = get_mysql_count(db_name, table_name)
    delta_count, delta_err = get_delta_count(delta_path)

    is_equal = (mysql_count is not None and delta_count is not None and mysql_count == delta_count)

    result_data.append(
        Row(
            **{
                "source_database": db_name,
                "source_table": table_name,
                "target_path": delta_path,
                "mysql_count": mysql_count,
                "delta_count": delta_count,
                "is_equal": is_equal,
                "mysql_error": mysql_err,
                "delta_error": delta_err,
            }
            # **{
            #     "source_database": db_name,
            #     "source_table": table_name,
            #     "target_path": delta_path,
            #     "mysql_count": None,
            #     "delta_count": delta_count,
            #     "is_equal": None,
            #     "mysql_error": None,
            #     "delta_error": delta_err,
            # }
        )
    )

result_schema = StructType([
    StructField("source_database", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_path", StringType(), True),
    StructField("mysql_count", LongType(), True),
    StructField("delta_count", LongType(), True),
    StructField("is_equal", BooleanType(), True),
    StructField("mysql_error", StringType(), True),
    StructField("delta_error", StringType(), True),
])

result_df = spark.createDataFrame(result_data, schema=result_schema)

# 展示结果
result_df.cache()
display(result_df)

# 如需保存
# result_df.write.mode("overwrite").parquet("abfss://.../check_result")
# 或导出 CSV（单文件）
# result_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("abfss://.../check_result_csv")

In [0]:
display(result_df.where(" delta_count = 0 or delta_count is null"))
result_df.unpersist()